# Model Error Analysis

This notebook analyzes model mistakes on the fake-news classification task. It reports confusion matrices, classification metrics, false positives, false negatives, and the relationship between text length and prediction errors.

In [1]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch_geometric.loader import DataLoader

sys.path.insert(0, os.path.abspath('..'))

from src.data.dataset_builder import load_raw_parquet_data, create_pyg_graphs_from_raw, split_datasets_strict
from src.data.augment_prune import augment_dataset, prune_dataset
from src.models.hyd_fake import HyDFakeModel

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'font.size': 11,
    'savefig.dpi': 150,
    'figure.dpi': 150,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


def load_raw_text(dataset: str, raw_dir: str = '../data/raw') -> pd.DataFrame:
    raw_path = Path(raw_dir) / dataset / 'raw_text'
    parquet_files = sorted(raw_path.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {raw_path}')
    df = pd.read_parquet(parquet_files[0]).copy()
    text_col = 'text' if 'text' in df.columns else df.columns[0]
    df[text_col] = df[text_col].astype(str).fillna('')
    df['text_length'] = df[text_col].str.split().str.len().fillna(0).astype('int16')
    df.attrs['text_col'] = text_col
    return df


def check_model_exists(dataset: str, output_dir: str = '../outputs/hyd_fake') -> bool:
    model_path = Path(output_dir) / dataset / 'model_hyd_fake_final.pt'
    return model_path.exists()


def generate_predictions_for_dataset(dataset: str, raw_dir: str = '../data/raw', interim_dir: str = '../data/interim', output_dir: str = '../outputs/hyd_fake') -> pd.DataFrame:
    model_path = Path(output_dir) / dataset / 'model_hyd_fake_final.pt'
    if not model_path.exists():
        raise FileNotFoundError(f'Model checkpoint not found: {model_path}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dict = load_raw_parquet_data(dataset, raw_dir=raw_dir, interim_dir=interim_dir, encoder='sbert')
    graphs = augment_dataset(create_pyg_graphs_from_raw(data_dict))
    if dataset == 'gossipcop':
        graphs = prune_dataset(graphs, max_depth=3)

    _, _, test_graphs = split_datasets_strict(graphs, dataset_name=dataset, raw_dir=raw_dir)
    if not test_graphs:
        raise ValueError(f'No test graphs found for dataset: {dataset}')

    loader = DataLoader(test_graphs, batch_size=16, shuffle=False)
    model = HyDFakeModel(
        text_frozen_dim=test_graphs[0].text_x.shape[1],
        graph_in_dim=test_graphs[0].x.shape[1],
        hidden_dim=256,
        num_gat_layers=4,
        edge_prune_threshold=0.05,
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device).eval()

    raw_df = load_raw_text(dataset, raw_dir=raw_dir)
    text_col = raw_df.attrs['text_col']

    rows = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.to(device))
            probabilities = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            predicted = logits.argmax(dim=1).detach().cpu().numpy()
            true_labels = batch.y.detach().cpu().numpy()
            graph_ids = batch.graph_id.detach().cpu().numpy() if hasattr(batch, 'graph_id') else np.arange(len(true_labels))

            for index, graph_id in enumerate(graph_ids):
                text_value = str(raw_df.iloc[int(graph_id)][text_col]) if int(graph_id) < len(raw_df) else ''
                rows.append({
                    'dataset': dataset,
                    'graph_id': int(graph_id),
                    'true_label': int(true_labels[index]),
                    'pred_label': int(predicted[index]),
                    'prob_fake': float(probabilities[index]),
                    'text': text_value,
                    'word_count': int(len(text_value.split())),
                })

    return pd.DataFrame(rows)


def load_or_build_predictions(prediction_path: str = '../results/predictions.csv', raw_dir: str = '../data/raw') -> pd.DataFrame:
    prediction_file = Path(prediction_path)
    
    # Case 1: Prediction file exists - load it
    if prediction_file.exists():
        print(f'✓ Loading predictions from {prediction_file}...')
        predictions = pd.read_csv(prediction_file, dtype={'true_label': 'int8', 'pred_label': 'int8', 'word_count': 'int16'})
        print(f'  Loaded {len(predictions):,} predictions')
        
    # Case 2: Prediction file doesn't exist - try to generate from models
    else:
        print(f'\n⚠ Predictions file not found: {prediction_file}')
        
        # Check if models exist
        models_exist = all(check_model_exists(dataset) for dataset in ['politifact', 'gossipcop'])
        
        if not models_exist:
            print(f'⚠ Model checkpoints not found in ../outputs/hyd_fake/')
            print(f'\nGenerating basic analysis from raw text data instead...\n')
            
            # Generate basic predictions from raw data (random or placeholder)
            frames = []
            for dataset in ['politifact', 'gossipcop']:
                print(f'  Loading raw text for {dataset}...')
                raw_df = load_raw_text(dataset, raw_dir=raw_dir)
                text_col = raw_df.attrs['text_col']
                
                # Create placeholder predictions (can be replaced with actual model outputs later)
                predictions_data = {
                    'dataset': dataset,
                    'graph_id': np.arange(len(raw_df)),
                    'true_label': np.random.randint(0, 2, len(raw_df)),  # Random labels as placeholder
                    'pred_label': np.random.randint(0, 2, len(raw_df)),  # Random predictions as placeholder
                    'prob_fake': np.random.rand(len(raw_df)),
                    'text': raw_df[text_col].values,
                    'word_count': raw_df['text_length'].values,
                }
                frames.append(pd.DataFrame(predictions_data))
                del raw_df
                gc.collect()
            
            predictions = pd.concat(frames, ignore_index=True)
            del frames
            gc.collect()
            
            print(f'\n⚠ WARNING: Using placeholder predictions. For real analysis, ensure:')
            print(f'   1. Model checkpoints exist in: ../outputs/hyd_fake/politifact/model_hyd_fake_final.pt')
            print(f'   2.                           ../outputs/hyd_fake/gossipcop/model_hyd_fake_final.pt')
            print(f'\n')
        
        else:
            print(f'✓ Model checkpoints found. Generating predictions from models...')
            prediction_file.parent.mkdir(parents=True, exist_ok=True)
            frames = []
            for dataset in ['politifact', 'gossipcop']:
                print(f'  Processing {dataset}...')
                try:
                    frame = generate_predictions_for_dataset(dataset)
                    frames.append(frame)
                    print(f'    Generated {len(frame):,} predictions for {dataset}')
                    gc.collect()
                except Exception as e:
                    print(f'    ERROR: Failed to generate predictions for {dataset}: {str(e)}')
                    continue
            
            if not frames:
                raise ValueError('Failed to generate predictions for any dataset')
            
            predictions = pd.concat(frames, ignore_index=True)
            predictions.to_csv(prediction_file, index=False)
            print(f'✓ Saved predictions to {prediction_file}')
            del frames
            gc.collect()

    # Standardize columns
    required_columns = {'true_label', 'pred_label', 'text'}
    missing_columns = required_columns - set(predictions.columns)
    if missing_columns:
        raise ValueError(f'Predictions missing required columns: {sorted(missing_columns)}')

    if 'dataset' not in predictions.columns:
        predictions['dataset'] = 'unknown'
    if 'prob_fake' not in predictions.columns:
        predictions['prob_fake'] = 0.5
    else:
        predictions['prob_fake'] = predictions['prob_fake'].fillna(0.5)
    
    if 'word_count' not in predictions.columns:
        predictions['word_count'] = predictions['text'].fillna('').astype(str).str.split().str.len().fillna(0).astype('int16')
    else:
        predictions['word_count'] = predictions['word_count'].fillna(0).astype('int16')

    predictions['true_label'] = predictions['true_label'].astype('int8')
    predictions['pred_label'] = predictions['pred_label'].astype('int8')
    predictions['is_error'] = predictions['true_label'] != predictions['pred_label']
    predictions['error_type'] = np.select(
        [
            (predictions['true_label'] == 0) & (predictions['pred_label'] == 1),
            (predictions['true_label'] == 1) & (predictions['pred_label'] == 0),
        ],
        ['false_positive', 'false_negative'],
        default='correct',
    )
    return predictions

d:\HyD-Fake\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Predictions and Overall Performance

We first load or generate the prediction table from HyDFake model checkpoints.
This section shows overall classification metrics, confusion matrix, and per-dataset breakdown.
All analyses are reproducible within this workspace.


In [ ]:
predictions = load_or_build_predictions('../results/predictions.csv', raw_dir='../data/raw')

print(f'\n{"="*60}')
print(f'OVERALL PERFORMANCE ANALYSIS')
print(f'{"="*60}')
print(f'Total predictions: {len(predictions):,}')
print(f'Unique datasets: {predictions["dataset"].nunique()}')
print(f'\nPrediction distribution:')
correct_count = (predictions['is_error'] == False).sum()
incorrect_count = (predictions['is_error'] == True).sum()
print(f'  Correct: {correct_count:,} ({100 * correct_count / len(predictions):.2f}%)')
print(f'  Incorrect: {incorrect_count:,} ({100 * incorrect_count / len(predictions):.2f}%)')

print(f'\nSample predictions (first 10 rows):')
display(predictions.head(10))

label_names = ['Real', 'Fake']
conf_matrix = confusion_matrix(predictions['true_label'], predictions['pred_label'])
report = classification_report(predictions['true_label'], predictions['pred_label'], target_names=label_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T

print('\nClassification Report')
display(report_df.round(4))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=label_names, yticklabels=label_names, ax=ax,
            cbar_kws={'label': 'Count'})
ax.set_title('Confusion Matrix - Overall Performance')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
plt.tight_layout()
plt.show()

# Per-dataset performance
print('\n\nPer-Dataset Performance:')
for dataset in sorted(predictions['dataset'].unique()):
    if pd.isna(dataset) or dataset == 'unknown':
        continue
    dataset_preds = predictions[predictions['dataset'] == dataset].copy()
    if len(dataset_preds) == 0:
        continue
    accuracy = (dataset_preds['true_label'] == dataset_preds['pred_label']).mean()
    print(f'\n{dataset.upper()}: {len(dataset_preds):,} samples, Accuracy: {100 * accuracy:.2f}%')
    
    ds_report = classification_report(dataset_preds['true_label'], dataset_preds['pred_label'], 
                                       target_names=label_names, output_dict=True, zero_division=0)
    ds_report_df = pd.DataFrame(ds_report).T
    display(ds_report_df.round(4))
    del dataset_preds
    gc.collect()


⚠ Predictions file not found: ..\results\predictions.csv
✓ Model checkpoints found. Generating predictions from models...
  Processing politifact...


Augmenting features: 100%|██████████| 314/314 [00:02<00:00, 152.11it/s]
C:\Users\ASUS ZENBOOK\AppData\Local\Temp\ipykernel_12908\2376914669.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this expe

    Generated 221 predictions for politifact
  Processing gossipcop...


Pruning graphs: 100%|██████████| 5464/5464 [00:14<00:00, 381.23it/s]
C:\Users\ASUS ZENBOOK\AppData\Local\Temp\ipykernel_12908\2376914669.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experim

## Detailed Error Analysis

This section investigates false positives, false negatives, and confidence calibration.
We examine text length patterns, confidence distribution, and identify the most challenging samples.
These insights reveal model weaknesses and guide improvements.


In [ ]:
print(f'\n{"="*60}')
print(f'DETAILED ERROR ANALYSIS')
print(f'{"="*60}')

false_positives = predictions[(predictions['true_label'] == 0) & (predictions['pred_label'] == 1)].copy()
false_negatives = predictions[(predictions['true_label'] == 1) & (predictions['pred_label'] == 0)].copy()
correct = predictions[predictions['is_error'] == False].copy()

print(f'\nError Summary:')
fp_count = len(false_positives)
fn_count = len(false_negatives)
correct_count = len(correct)
print(f'  False positives (real → fake): {fp_count:,} ({100*fp_count/len(predictions):.2f}%)')
print(f'  False negatives (fake → real): {fn_count:,} ({100*fn_count/len(predictions):.2f}%)')
print(f'  Correct predictions: {correct_count:,} ({100*correct_count/len(predictions):.2f}%)')

if len(false_positives) > 0:
    print(f'\nFalse Positives Examples (Real articles misclassified as Fake):')
    fp_sample = false_positives.nlargest(5, 'prob_fake')[['dataset', 'word_count', 'prob_fake', 'text']]
    display(fp_sample)
else:
    print('No false positives found')

if len(false_negatives) > 0:
    print(f'\nFalse Negatives Examples (Fake articles misclassified as Real):')
    fn_sample = false_negatives.nsmallest(5, 'prob_fake')[['dataset', 'word_count', 'prob_fake', 'text']]
    display(fn_sample)
else:
    print('No false negatives found')

# Text length analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Box plot: word count vs error type
sns.boxplot(data=predictions, x='error_type', y='word_count', ax=axes[0, 0], 
            palette=['#2E86AB', '#E67E22', '#27AE60'],
            order=['correct', 'false_positive', 'false_negative'])
axes[0, 0].set_title('Word Count Distribution by Error Type')
axes[0, 0].set_xlabel('Error Type')
axes[0, 0].set_ylabel('Word count')
axes[0, 0].set_xticklabels(['Correct', 'False Pos', 'False Neg'])

# Distribution: word count by error flag  
valid_word_counts = predictions[predictions['word_count'] > 0]
if len(valid_word_counts) > 0:
    sns.histplot(data=valid_word_counts, x='word_count', hue='is_error', bins=50, kde=True, 
                 stat='density', common_norm=False, ax=axes[0, 1],
                 palette=['#2E86AB', '#E67E22'])
axes[0, 1].set_title('Word Count Distribution by Correctness')
axes[0, 1].set_xlabel('Word count')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend(['Correct', 'Error'], title='Prediction')

# Confidence by error type
sns.violinplot(data=predictions, x='error_type', y='prob_fake', ax=axes[1, 0],
               palette=['#2E86AB', '#E67E22', '#27AE60'],
               order=['correct', 'false_positive', 'false_negative'])
axes[1, 0].set_title('Model Confidence by Error Type')
axes[1, 0].set_xlabel('Error Type')
axes[1, 0].set_ylabel('P(Fake)')
axes[1, 0].set_xticklabels(['Correct', 'False Pos', 'False Neg'])

# Error rate by dataset
ds_errors = predictions[predictions['dataset'].notna()].groupby('dataset').apply(lambda x: {
    'fp': ((x['true_label'] == 0) & (x['pred_label'] == 1)).sum(),
    'fn': ((x['true_label'] == 1) & (x['pred_label'] == 0)).sum(),
    'total': len(x)
}).apply(pd.Series)
ds_errors['fp_rate'] = 100 * ds_errors['fp'] / ds_errors['total']
ds_errors['fn_rate'] = 100 * ds_errors['fn'] / ds_errors['total']

if len(ds_errors) > 0:
    ds_errors[['fp_rate', 'fn_rate']].plot(kind='bar', ax=axes[1, 1], color=['#E67E22', '#27AE60'])
    axes[1, 1].set_title('Error Rate by Dataset')
    axes[1, 1].set_xlabel('Dataset')
    axes[1, 1].set_ylabel('Error Rate (%)')
    axes[1, 1].legend(['False Positive Rate', 'False Negative Rate'])
    axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Confidence calibration analysis
print('\n\nConfidence Calibration:')
valid_probs = predictions[predictions['prob_fake'].notna()]['prob_fake']
if len(valid_probs) > 0:
    confidence_bins = pd.cut(valid_probs, bins=[0, 0.25, 0.5, 0.75, 1.0], 
                             labels=['0-25%', '25-50%', '50-75%', '75-100%'])
    calibration_data = []
    for bin_label in confidence_bins.cat.categories:
        bin_mask = confidence_bins == bin_label
        if bin_mask.any():
            bin_indices = predictions[predictions['prob_fake'].notna()].index[bin_mask]
            bin_preds = predictions.loc[bin_indices]
            accuracy = (bin_preds['true_label'] == bin_preds['pred_label']).mean()
            fake_count = (bin_preds['true_label'] == 1).sum()
            total_count = len(bin_preds)
            calibration_data.append({
                'Confidence Range': bin_label,
                'Sample Count': total_count,
                'Fake Percentage': 100 * fake_count / total_count if total_count > 0 else 0,
                'Accuracy': 100 * accuracy if total_count > 0 else 0,
                'Avg Prob': bin_preds['prob_fake'].mean()
            })
    
    if calibration_data:
        calibration_df = pd.DataFrame(calibration_data)
        display(calibration_df.round(2))

# Most ambiguous samples
ambiguous_samples = predictions.assign(uncertainty=(predictions['prob_fake'] - 0.5).abs()).sort_values('uncertainty').head(15)
print('\nMost Ambiguous Samples (Confidence closest to 50%):')  
display(ambiguous_samples[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'is_error']])

# Shortest and longest problematic samples
if (predictions['is_error'] == True).sum() > 0:
    print('\nShortest Problematic Samples:')
    short_errors = predictions[predictions['is_error'] == True].nsmallest(10, 'word_count')
    display(short_errors[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'text']])
    
    print('\nLongest Problematic Samples:')
    long_errors = predictions[predictions['is_error'] == True].nlargest(10, 'word_count')
    display(long_errors[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'text']])
else:
    print('No errors found for analysis')

# Cleanup
del false_positives, false_negatives, correct
gc.collect()
print('\nMemory cleanup complete')

## Detailed Error Analysis and Insights

### Error Type Distribution

**False Positives** (Real → Fake): The model incorrectly flags legitimate news as fake. Common causes:
- Sensational wording or emphasis in real reporting
- Emotional language or strong claims that mimic fake-news style
- Breaking news or controversial topics that trigger higher fake confidence
- Limited context from short articles

**False Negatives** (Fake → Real): The model fails to detect genuine misinformation. Common causes:
- Sophisticated misinformation that mimics credible reporting style
- High-quality writing that avoids typical fake-news markers
- Ambiguous claims that aren't definitively false
- Limited network evidence (few shares, isolated cascades)

**Correct Predictions**: The model successfully identifies both real and fake articles with high confidence and/or clear structural signals.

### Text Length Analysis

1. **Word Count Distribution**
   - Shorter articles (< 100 words) tend to have higher error rates due to insufficient context
   - Very long articles (> 1000 words) may dilute the strongest semantic cues
   - Optimal range for model accuracy is typically 100-500 words

2. **Length-Based Error Patterns**
   - Short false positives: Real news with emotional headlines but limited body text
   - Short false negatives: Fake news presented very concisely without typical markers
   - Long false positives: Real in-depth reporting with charged language
   - Long false negatives: Sophisticated fake news that uses professional writing style

### Confidence Calibration

- **Well-calibrated model**: Confidence matches actual accuracy
- **Overconfident**: Model gives high confidence but makes errors
- **Underconfident**: Model is uncertain even when predictions are correct
- **Ambiguous samples**: Confidence near 50% indicates genuinely difficult cases

### Per-Dataset Performance

Different datasets (Politifact vs GossipCop) may have different error patterns:
- Different rumor types may be harder to classify
- Dataset-specific linguistic patterns affect model behavior
- Graph structure differences influence network-based decisions

### Recommendations

1. **For production deployment**: Use confidence threshold to flag uncertain predictions for human review
2. **For model improvement**: 
   - Augment training data with difficult samples (near-confidence examples)
   - Investigate feature importance for high-error-rate samples
   - Consider dataset-specific fine-tuning
3. **For interpretation**: Combined text + graph signals are essential; text alone leaves ambiguous cases unresolved
